# EGIER Bag Production Trend Analysis

**Mathematical Modeling & Numerical Methods Portfolio Project**

This project analyzes monthly EGIER bag production data from **January 2018 to December 2023** and uses mathematical curve fitting to forecast when production may reach the current warehouse capacity.

### Objectives

- Explore the historical production trend.
- Fit and compare Quadratic, Cubic, Exponential, and Logistic models.
- Select the best model using **Root Mean Square Error (RMSE)**.
- Forecast when production reaches a warehouse capacity of **25,000 units**.
- Estimate the latest recommended construction start date using a **13-month construction lead time**.
- Validate numerical calculations using polynomial evaluation and Taylor approximation.

> **Important:** Forecasts beyond the observed 2018–2023 period are model-based estimates, not guarantees of future production.


## 1. Setup

The analysis uses NumPy for numerical computation, SciPy for nonlinear curve fitting and root finding, and Matplotlib for visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, brentq

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True


## 2. Dataset

The dataset contains 144 monthly production observations. Month 1 represents January 2018 and Month 144 represents December 2023.

In [ ]:
production = np.array([1863, 1614, 2570, 1685, 2101, 1811, 2457, 2171, 2134, 2502, 2358, 2399, 2048, 2523, 2086, 2391, 2150, 2340, 3129, 2277, 2964, 2997, 2747, 2862, 3405, 2677, 2749, 2755, 2963, 3161, 3623, 2768, 3141, 3439, 3601, 3531, 3477, 3376, 4027, 3175, 3274, 3334, 3964, 3649, 3502, 3688, 3657, 4422, 4197, 4441, 4736, 4521, 4485, 4644, 5036, 4876, 4789, 4544, 4975, 5211, 4880, 4933, 5079, 5339, 5232, 5520, 5714, 5260, 6110, 5334, 5988, 6235, 6365, 6266, 6345, 6118, 6497, 6278, 6638, 6590, 6271, 7246, 6584, 6594, 7092, 7326, 7409, 7976, 7959, 8012, 8195, 8008, 8313, 7791, 8368, 8933, 8756, 8613, 8705, 9098, 8769, 9544, 9050, 9186, 10012, 9685, 9966, 10048, 10244, 10740, 10318, 10393, 10986, 10635, 10731, 11749, 11849, 12123, 12274, 11666, 11960, 12629, 12915, 13051, 13387, 13309, 13732, 13162, 13644, 13808, 14101, 13992, 15191, 15018, 14917, 15046, 15556, 15893, 16388, 16782, 16716, 17033, 16896, 17689], dtype=float)
months = np.arange(1, len(production) + 1)

print(f"Observations: {len(production)} months")
print(f"Period: January 2018 - December 2023")
print(f"First recorded production: {production[0]:,.0f} units")
print(f"Last recorded production: {production[-1]:,.0f} units")


In [ ]:
plt.figure()
plt.plot(months, production, marker="o", markersize=2, linewidth=1)
plt.title("EGIER Monthly Bag Production")
plt.xlabel("Month (M1 = Jan 2018)")
plt.ylabel("Bags Produced")
plt.tight_layout()
plt.show()


## 3. Model Definitions

Four candidate models are evaluated:

1. **Quadratic**
2. **Cubic**
3. **Exponential**
4. **Logistic**

RMSE is used as the model-selection metric. A lower RMSE means the model has a smaller average fitting error over the historical observations.

In [ ]:
def quadratic_model(x, a, b, c):
    return a * x**2 + b * x + c

def cubic_model(x, a, b, c, d):
    return a * x**3 + b * x**2 + c * x + d

def exponential_model(x, a, b, c):
    return a * np.exp(b * x) + c

def logistic_model(x, a, b, c, d):
    return a / (1 + np.exp(-b * (x - c))) + d

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


## 4. Fit and Compare Models

Each model is fitted to the complete historical dataset. The Logistic model uses bounded parameters to avoid numerical overflow and unstable optimization.

In [ ]:
models = {}

# Polynomial fits
quad_params = np.polyfit(months, production, 2)
cubic_params = np.polyfit(months, production, 3)

models["Quadratic"] = {
    "function": quadratic_model,
    "params": quad_params,
    "prediction": quadratic_model(months, *quad_params),
}

models["Cubic"] = {
    "function": cubic_model,
    "params": cubic_params,
    "prediction": cubic_model(months, *cubic_params),
}

# Exponential fit
exp_params, _ = curve_fit(
    exponential_model,
    months,
    production,
    p0=[1000, 0.01, 0],
    maxfev=20000
)

models["Exponential"] = {
    "function": exponential_model,
    "params": exp_params,
    "prediction": exponential_model(months, *exp_params),
}

# Logistic fit with sensible bounds
logistic_params, _ = curve_fit(
    logistic_model,
    months,
    production,
    p0=[20000, 0.03, 100, 0],
    bounds=([0, 0, 1, -50000], [100000, 1, 300, 50000]),
    maxfev=50000
)

models["Logistic"] = {
    "function": logistic_model,
    "params": logistic_params,
    "prediction": logistic_model(months, *logistic_params),
}

for name, result in models.items():
    result["rmse"] = rmse(production, result["prediction"])

results = sorted(
    [(name, result["rmse"]) for name, result in models.items()],
    key=lambda x: x[1]
)

print("Model comparison:")
for name, score in results:
    print(f"{name:12s} RMSE = {score:,.2f}")

best_name = results[0][0]
best = models[best_name]

print(f"\nSelected model: {best_name}")


In [ ]:
x_fit = np.linspace(1, 144, 300)

plt.figure()
plt.scatter(months, production, s=15, alpha=0.6, label="Actual Production")

for name, result in models.items():
    plt.plot(
        x_fit,
        result["function"](x_fit, *result["params"]),
        linewidth=2,
        label=f"{name} (RMSE={result['rmse']:.1f})"
    )

plt.title("Comparison of Production Models")
plt.xlabel("Month")
plt.ylabel("Bags Produced")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Capacity Forecast

The warehouse capacity is set to **25,000 units**. The capacity-crossing month is found by solving:

\[
f(t) = 25,000
\]

A bracketing root finder (`brentq`) is used instead of relying on a single initial guess. This makes the capacity calculation more robust for the selected model.

In [ ]:
capacity = 25_000
construction_lead_time = 13  # months

def model_value(t):
    return best["function"](t, *best["params"])

# Find a sign-changing interval beyond the observed period.
search_months = np.linspace(144, 1000, 5000)
values = model_value(search_months) - capacity

crossing_indices = np.where(values[:-1] * values[1:] <= 0)[0]

if len(crossing_indices) == 0:
    raise RuntimeError("No capacity crossing found in the search range.")

i = crossing_indices[0]
critical_month = brentq(
    lambda t: model_value(t) - capacity,
    search_months[i],
    search_months[i + 1]
)

construction_start = critical_month - construction_lead_time

print(f"Selected model: {best_name}")
print(f"Warehouse capacity: {capacity:,.0f} units")
print(f"Capacity reached at approximately Month {critical_month:.2f}")
print(f"Recommended construction start: approximately Month {construction_start:.2f}")


In [ ]:
def month_to_date(month_number):
    month_number = float(month_number)
    year = 2018 + int((month_number - 1) // 12)
    month = int((month_number - 1) % 12) + 1
    month_names = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]
    return f"{month_names[month - 1]} {year}"

print("=== Capacity Planning Result ===")
print(f"Production reaches {capacity:,.0f} units: {month_to_date(critical_month)}")
print(f"Construction should start around: {month_to_date(construction_start)}")
print(f"Construction lead time: {construction_lead_time} months")


In [ ]:
forecast_months = np.linspace(1, max(critical_month + 15, 160), 500)

plt.figure()
plt.scatter(months, production, s=15, alpha=0.55, label="Historical Production")
plt.plot(
    forecast_months,
    model_value(forecast_months),
    linewidth=2,
    label=f"{best_name} Model"
)
plt.axhline(capacity, linestyle="--", linewidth=1.5, label="Warehouse Capacity")
plt.axvline(
    critical_month,
    linestyle=":",
    linewidth=1.5,
    label=f"Capacity: {month_to_date(critical_month)}"
)
plt.axvline(
    construction_start,
    linestyle=":",
    linewidth=1.5,
    label=f"Construction Start: {month_to_date(construction_start)}"
)

plt.title("Production Trend and Warehouse Capacity Forecast")
plt.xlabel("Month (M1 = Jan 2018)")
plt.ylabel("Bags Produced")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Numerical Approximation

For the selected model, polynomial evaluation can be performed efficiently using **Horner's method** when the model is polynomial. To keep the numerical-method component independent from model selection, the cubic fit is also evaluated using a third-order Taylor expansion around Month 72.

The comparison below measures the maximum absolute difference between the direct cubic calculation, Horner evaluation, and Taylor approximation over the historical period.

In [ ]:
a, b, c, d = cubic_params

def cubic_horner(x):
    return ((a * x + b) * x + c) * x + d

def cubic_taylor(x, center=72):
    h = x - center
    f = cubic_model(center, a, b, c, d)
    df = 3 * a * center**2 + 2 * b * center + c
    ddf = 6 * a * center + 2 * b
    dddf = 6 * a

    return (
        f
        + df * h
        + 0.5 * ddf * h**2
        + (1 / 6) * dddf * h**3
    )

test_months = np.linspace(1, 144, 100)
direct = cubic_model(test_months, a, b, c, d)
horner = cubic_horner(test_months)
taylor = cubic_taylor(test_months)

horner_error = np.max(np.abs(direct - horner))
taylor_error = np.max(np.abs(direct - taylor))

print(f"Maximum Horner absolute error: {horner_error:.10f}")
print(f"Maximum Taylor absolute error: {taylor_error:.10f}")


In [ ]:
plt.figure()
plt.plot(test_months, direct - horner, label="Horner Error")
plt.plot(test_months, direct - taylor, label="Taylor Error")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Numerical Approximation Error")
plt.xlabel("Month")
plt.ylabel("Error (units)")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Validation and Limitations

### Validation

The final model should be checked against the latest observed production value before being used for planning. This prevents an apparently good curve fit from being treated as automatically reliable for extrapolation.

### Limitations

- The forecast extends beyond the 2018–2023 observation period, so uncertainty increases with forecast horizon.
- RMSE measures historical fitting error; it does not guarantee good future forecasting performance.
- Production may be affected by demand, supply, operational constraints, or other factors not represented in the model.
- Warehouse planning should be revisited as new production data becomes available.
- The construction-start date is a planning estimate based on the assumed 13-month lead time.

## 8. Conclusion

This project demonstrates a complete workflow for **trend analysis, mathematical model comparison, capacity forecasting, and numerical-method validation** using Python.

The final capacity date is determined programmatically from the selected model rather than hard-coding a result, making the notebook easier to reproduce and update when new data becomes available.


## References / Project Context

This notebook is based on the EGIER bag-production dataset and the warehouse-capacity planning scenario used in the original analysis.

**Repository:** `parmesanpizzoy/Trend-Analysis`
